In [ ]:
!pip install -q \
    datasets \
    langchain \
    langchain-core \
    langchain-chroma \
    langchain-huggingface \
    chromadb \
    transformers \
    accelerate \
    sentence-transformers

In [ ]:
import os
import re
import zipfile
import random
import torch
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm
from google.colab import files

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
uploaded = files.upload()

if not uploaded:
    raise ValueError(
        "No ZIP file was uploaded."
    )

uploaded_filename = next(iter(uploaded))

ZIP_PATH = Path(
    uploaded_filename
).resolve()

print("Uploaded file:", ZIP_PATH)
print("File exists:", ZIP_PATH.exists())

EXTRACT_DIRECTORY = (
    Path.cwd() / "radiology_kb"
)

EXTRACT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as zip_file:

    zip_file.extractall(
        EXTRACT_DIRECTORY
    )

CHROMA_DIRECTORY = (
    EXTRACT_DIRECTORY / "chroma_db"
)

if not CHROMA_DIRECTORY.exists():
    raise FileNotFoundError(
        f"ChromaDB folder not found: "
        f"{CHROMA_DIRECTORY}"
    )

print("Extraction successful")
print(
    "ChromaDB directory:",
    CHROMA_DIRECTORY
)
print(
    "Database files:",
    os.listdir(CHROMA_DIRECTORY)
)

Saving biolaysumm_rrg_chromadb.zip to biolaysumm_rrg_chromadb (2).zip
Uploaded file: /content/biolaysumm_rrg_chromadb (2).zip
File exists: True
Extraction successful
ChromaDB directory: /content/radiology_kb/chroma_db
Database files: ['d3072495-7243-4e49-bb8c-1f03612f8b94', 'chroma.sqlite3']


In [ ]:
from langchain_huggingface import (
    HuggingFaceEmbeddings
)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Embedding device:", DEVICE)

embedding_model = HuggingFaceEmbeddings(
    model_name=(
        "sentence-transformers/"
        "all-MiniLM-L6-v2"
    ),
    model_kwargs={
        "device": DEVICE
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

print("Embedding model loaded.")

Embedding device: cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.


In [ ]:
kb_data = (
    vector_store
    ._collection
    .get(
        include=[
            "documents",
            "metadatas"
        ]
    )
)

KB_ENTRIES = []

for document, metadata in zip(
    kb_data["documents"],
    kb_data["metadatas"]
):
    KB_ENTRIES.append({
        "term": metadata.get(
            "term",
            ""
        ),
        "definition": document,
        "metadata": metadata
    })

print(
    "Knowledge-base entries loaded:",
    len(KB_ENTRIES)
)

assert len(KB_ENTRIES) == 210

Knowledge-base entries loaded: 210


In [ ]:
kb_data = (
    vector_store
    ._collection
    .get(
        include=[
            "documents",
            "metadatas"
        ]
    )
)

KB_ENTRIES = []

for document, metadata in zip(
    kb_data["documents"],
    kb_data["metadatas"]
):
    KB_ENTRIES.append({
        "term": metadata.get(
            "term",
            ""
        ),
        "definition": document,
        "metadata": metadata
    })

print(
    "Knowledge-base entries loaded:",
    len(KB_ENTRIES)
)

assert len(KB_ENTRIES) == 210

Knowledge-base entries loaded: 210


In [ ]:
import re


EXCLUDED_RETRIEVAL_TERMS = {
    "no evidence of",
    "without",
    "cannot exclude",
    "may represent",
    "possible",
    "probable",
    "stable",
    "unchanged",
    "interval increase",
    "interval decrease",
    "improved",
    "resolved",
    "new",
    "acute"
}


def normalize_text(text):
    text = str(text).lower()

    text = re.sub(
        r"[^a-z0-9\s-]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def term_occurs_in_report(
    term,
    report
):
    normalized_term = normalize_text(
        term
    )

    normalized_report = normalize_text(
        report
    )

    if normalized_term.endswith("s"):
        plural_ending = ""
    else:
        plural_ending = "s?"

    pattern = (
        r"(?<![a-z0-9])"
        + re.escape(normalized_term)
        + plural_ending
        + r"(?![a-z0-9])"
    )

    return bool(
        re.search(
            pattern,
            normalized_report
        )
    )


def retrieve_definitions(
    report,
    vector_store,
    maximum_results=5,
    fallback_threshold=0.75
):
    exact_candidates = []

    for entry in KB_ENTRIES:

        term = entry["term"]

        if not term:
            continue

        normalized_term = normalize_text(
            term
        )

        if (
            normalized_term
            in EXCLUDED_RETRIEVAL_TERMS
        ):
            continue

        if term_occurs_in_report(
            term,
            report
        ):
            exact_candidates.append({
                "term": term,
                "definition":
                    entry["definition"],
                "score": 1.0,
                "match_type": "exact"
            })

    exact_candidates.sort(
        key=lambda item: len(
            normalize_text(
                item["term"]
            )
        ),
        reverse=True
    )

    exact_results = []
    selected_terms = []

    for candidate in exact_candidates:

        candidate_term = normalize_text(
            candidate["term"]
        )

        contained_in_longer_term = any(
            candidate_term
            in selected_term
            for selected_term
            in selected_terms
        )

        if contained_in_longer_term:
            continue

        exact_results.append(
            candidate
        )

        selected_terms.append(
            candidate_term
        )

        if (
            len(exact_results)
            >= maximum_results
        ):
            break

    if exact_results:
        return exact_results

    vector_results = (
        vector_store
        .similarity_search_with_relevance_scores(
            report,
            k=5
        )
    )

    fallback_results = []

    for document, score in vector_results:

        if score < fallback_threshold:
            continue

        term = document.metadata.get(
            "term",
            ""
        )

        if not term:
            continue

        normalized_term = normalize_text(
            term
        )

        if (
            normalized_term
            in EXCLUDED_RETRIEVAL_TERMS
        ):
            continue

        fallback_results.append({
            "term": term,
            "definition":
                document.page_content,
            "score": float(score),
            "match_type": "semantic"
        })

    return fallback_results[:2]


print(
    "retrieve_definitions is now defined."
)

retrieve_definitions is now defined.


In [ ]:
print(retrieve_definitions)

<function retrieve_definitions at 0x7e14f261aa20>


In [ ]:
test_report = """
The chest shows significant air trapping.
Bilateral apical chronic changes are present.
Dorsal kyphosis is noted.
No evidence of pneumothorax.
"""

retrieved_entries = retrieve_definitions(
    report=test_report,
    vector_store=vector_store,
    maximum_results=5,
    fallback_threshold=0.75
)

print("Retrieved entries:")

for entry in retrieved_entries:
    print("\nTerm:", entry["term"])
    print("Match:", entry["match_type"])
    print("Score:", entry["score"])
    print(
        "Definition:",
        entry["definition"]
    )

Retrieved entries:

Term: dorsal kyphosis
Match: exact
Score: 1.0
Definition: Term: dorsal kyphosis. Meaning: Increased forward curvature of the thoracic spine.

Term: air trapping
Match: exact
Score: 1.0
Definition: Term: air trapping. Meaning: Air remains in part of the lungs during breathing out.

Term: pneumothorax
Match: exact
Score: 1.0
Definition: Term: pneumothorax. Meaning: Air has entered the space around the lung and may cause the lung to partially or completely collapse.

Term: chronic
Match: exact
Score: 1.0
Definition: Term: chronic. Meaning: Long-standing or present over an extended period.


In [ ]:
def retrieve_definitions(
    report,
    vector_store,
    maximum_results=5,
    fallback_threshold=0.75
):
    exact_candidates = []

    # Search every knowledge-base term.
    for entry in KB_ENTRIES:

        term = entry["term"]

        if not term:
            continue

        normalized_term = normalize_text(
            term
        )

        if (
            normalized_term
            in EXCLUDED_RETRIEVAL_TERMS
        ):
            continue

        if term_occurs_in_report(
            term,
            report
        ):
            exact_candidates.append({
                "term": term,
                "definition":
                    entry["definition"],
                "score": 1.0,
                "match_type": "exact"
            })

    # Prefer longer, specific terms.
    exact_candidates.sort(
        key=lambda item: len(
            normalize_text(
                item["term"]
            )
        ),
        reverse=True
    )

    exact_results = []
    selected_terms = []

    for candidate in exact_candidates:

        candidate_term = normalize_text(
            candidate["term"]
        )

        # Avoid selecting both:
        # dorsal kyphosis and kyphosis.
        contained_in_longer_term = any(
            candidate_term
            in selected_term
            for selected_term
            in selected_terms
        )

        if contained_in_longer_term:
            continue

        exact_results.append(
            candidate
        )

        selected_terms.append(
            candidate_term
        )

        if (
            len(exact_results)
            >= maximum_results
        ):
            break

    if exact_results:
        return exact_results

    # Vector fallback only when
    # no exact term is detected.
    vector_results = (
        vector_store
        .similarity_search_with_relevance_scores(
            report,
            k=5
        )
    )

    fallback_results = []

    for document, score in vector_results:

        if score < fallback_threshold:
            continue

        term = document.metadata.get(
            "term",
            ""
        )

        if not term:
            continue

        normalized_term = normalize_text(
            term
        )

        if (
            normalized_term
            in EXCLUDED_RETRIEVAL_TERMS
        ):
            continue

        fallback_results.append({
            "term": term,
            "definition":
                document.page_content,
            "score": float(score),
            "match_type": "semantic"
        })

    return fallback_results[:2]

print("Retriever loaded.")

Retriever loaded.


In [ ]:
test_report = """
The chest shows significant air trapping.
Bilateral apical chronic changes are present.
Dorsal kyphosis is noted.
No evidence of pneumothorax.
"""

retrieved_entries = retrieve_definitions(
    report=test_report,
    vector_store=vector_store,
    maximum_results=5,
    fallback_threshold=0.75
)

print("RETRIEVED TERMS")

for entry in retrieved_entries:
    print(
        f"- {entry['term']} "
        f"({entry['match_type']}): "
        f"{entry['score']:.4f}"
    )

    print(
        " ",
        entry["definition"]
    )

RETRIEVED TERMS
- dorsal kyphosis (exact): 1.0000
  Term: dorsal kyphosis. Meaning: Increased forward curvature of the thoracic spine.
- air trapping (exact): 1.0000
  Term: air trapping. Meaning: Air remains in part of the lungs during breathing out.
- pneumothorax (exact): 1.0000
  Term: pneumothorax. Meaning: Air has entered the space around the lung and may cause the lung to partially or completely collapse.
- chronic (exact): 1.0000
  Term: chronic. Meaning: Long-standing or present over an extended period.


In [ ]:
from huggingface_hub import login
from getpass import getpass

new_hf_token = getpass(
    "Paste your NEW Hugging Face token: "
)

login(
    token=new_hf_token,
    add_to_git_credential=False
)

print("Hugging Face login completed.")

Paste your NEW Hugging Face token: ··········
Hugging Face login completed.


In [ ]:
from huggingface_hub import whoami

account = whoami()

print(
    "Logged in as:",
    account["name"]
)

Logged in as: Aswathy-04


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

MODEL_NAME = (
    "meta-llama/"
    "Llama-3.2-1B-Instruct"
)

tokenizer = (
    AutoTokenizer
    .from_pretrained(
        MODEL_NAME
    )
)

tokenizer.clean_up_tokenization_spaces = (
    False
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = (
        tokenizer.eos_token
    )

model = (
    AutoModelForCausalLM
    .from_pretrained(
        MODEL_NAME,
        torch_dtype=(
            torch.float16
            if torch.cuda.is_available()
            else torch.float32
        ),
        device_map="auto",
        low_cpu_mem_usage=True
    )
)

model.eval()

print(
    "Llama 3.2 1B loaded successfully."
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Llama 3.2 1B loaded successfully.


In [ ]:
def format_context(
    retrieved_entries
):
    if not retrieved_entries:
        return (
            "No relevant glossary "
            "definitions were retrieved."
        )

    context_lines = []

    for entry in retrieved_entries:
        context_lines.append(
            f"- {entry['definition']}"
        )

    return "\n".join(
        context_lines
    )

print("Context formatter loaded.")

Context formatter loaded.


In [ ]:
def create_sentence_prompt(
    report_sentence,
    retrieved_context
):
    system_message = """
Rewrite one radiology-report sentence in simple,
patient-friendly language.

Rules:
1. Preserve every finding, negation, uncertainty,
   severity, anatomical location and comparison.
2. Use definitions only to simplify medical terms
   present in the original sentence.
3. Do not add a diagnosis, cause, symptom, duration,
   treatment, recommendation or medical history.
4. Do not omit any information from the sentence.
5. Do not explain your work or repeat the original.
6. Return exactly one simplified sentence without
   a heading, label, list, note or definition.
""".strip()

    user_message = f"""
ORIGINAL:
{report_sentence}

DEFINITIONS:
{retrieved_context}

SIMPLIFIED SENTENCE:
""".strip()

    messages = [
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


print(
    "Short Version 3 prompt loaded."
)

Short Version 3 prompt loaded.


In [ ]:
def generate_summary_from_prompt(
    prompt,
    max_new_tokens=45
):
    model_inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    model_inputs = {
        key: value.to(model.device)
        for key, value
        in model_inputs.items()
    }

    with torch.no_grad():

        generated_output = model.generate(
            **model_inputs,
            max_new_tokens=
                max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=
                tokenizer.eos_token_id,
            eos_token_id=
                tokenizer.eos_token_id
        )

    input_length = (
        model_inputs["input_ids"]
        .shape[1]
    )

    generated_tokens = (
        generated_output[
            0,
            input_length:
        ]
    )

    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    return generated_text.strip()


print(
    "Version 3 generation loaded."
)

Version 3 generation loaded.


In [ ]:
def generate_summary_from_prompt(
    prompt,
    max_new_tokens=45
):
    model_inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    model_inputs = {
        key: value.to(model.device)
        for key, value
        in model_inputs.items()
    }

    with torch.no_grad():

        generated_output = model.generate(
            **model_inputs,
            max_new_tokens=
                max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=
                tokenizer.eos_token_id,
            eos_token_id=
                tokenizer.eos_token_id
        )

    input_length = (
        model_inputs["input_ids"]
        .shape[1]
    )

    generated_tokens = (
        generated_output[
            0,
            input_length:
        ]
    )

    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    return generated_text.strip()


print(
    "Version 3 generation loaded."
)

Version 3 generation loaded.


In [ ]:
def split_report_sentences(
    report
):
    report = str(report).strip()

    sentences = re.split(
        r"(?<=[.!?])\s+",
        report
    )

    sentences = [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]

    return sentences

print("Sentence splitter loaded.")

Sentence splitter loaded.


In [ ]:
def generate_rag_summary(
    radiology_report,
    show_retrieval=True
):
    radiology_report = str(
        radiology_report
    ).strip()

    if not radiology_report:
        raise ValueError(
            "Radiology report cannot be empty."
        )

    report_sentences = (
        split_report_sentences(
            radiology_report
        )
    )

    simplified_sentences = []
    all_retrieved_entries = []

    for report_sentence in report_sentences:

        retrieved_entries = (
            retrieve_definitions(
                report=report_sentence,
                vector_store=vector_store,
                maximum_results=4,
                fallback_threshold=0.75
            )
        )

        all_retrieved_entries.extend(
            retrieved_entries
        )

        retrieved_context = (
            format_context(
                retrieved_entries
            )
        )

        prompt = create_sentence_prompt(
            report_sentence=
                report_sentence,
            retrieved_context=
                retrieved_context
        )

        generated_sentence = (
            generate_summary_from_prompt(
                prompt,
                max_new_tokens=45
            )
        )

        cleaned_sentence = (
            clean_generated_text(
                generated_sentence
            )
        )

        if cleaned_sentence:
            simplified_sentences.append(
                cleaned_sentence
            )

    summary = " ".join(
        simplified_sentences
    )

    unique_entries = []
    seen_terms = set()

    for entry in all_retrieved_entries:

        normalized_term = normalize_text(
            entry["term"]
        )

        if normalized_term in seen_terms:
            continue

        seen_terms.add(
            normalized_term
        )

        unique_entries.append(
            entry
        )

    if show_retrieval:

        print("RETRIEVED TERMS")

        if not unique_entries:
            print(
                "No definitions retrieved."
            )

        for entry in unique_entries:
            print(
                f"- {entry['term']} "
                f"({entry['match_type']}): "
                f"{entry['score']:.4f}"
            )

        print("\nRAG SUMMARY")
        print(summary)

    return {
        "radiology_report":
            radiology_report,

        "retrieved_terms": [
            entry["term"]
            for entry
            in unique_entries
        ],

        "retrieval_scores": [
            entry["score"]
            for entry
            in unique_entries
        ],

        "match_types": [
            entry["match_type"]
            for entry
            in unique_entries
        ],

        "summary": summary
    }


print(
    "Version 3 RAG function loaded."
)

Version 3 RAG function loaded.


In [ ]:
test_prompt = create_sentence_prompt(
    report_sentence=(
        "Chronic changes are present."
    ),
    retrieved_context=(
        "Chronic means long-term."
    )
)

assert (
    "Never invent a duration"
    in test_prompt
)

assert (
    clean_generated_text(
        "No pneumohorax is present."
    )
    ==
    "No pneumothorax is present."
)

print(
    "Corrected functions are active."
)

Corrected functions are active.


In [ ]:
test_report = """
The chest shows significant air trapping.
Bilateral apical chronic changes are present.
Dorsal kyphosis is noted.
No evidence of pneumothorax.
"""

result = generate_rag_summary(
    test_report,
    show_retrieval=True
)

RETRIEVED TERMS
- air trapping (exact): 1.0000
- chronic (exact): 1.0000
- dorsal kyphosis (exact): 1.0000
- pneumothorax (exact): 1.0000

RAG SUMMARY
The patient has difficulty exhaling air from their lungs. Bilateral apical changes refer to long-standing abnormalities in the heart's upper chambers. The patient has a forward curvature of the upper back. Patient has air in the space around the lung, which is causing the lung to partially or completely collapse.


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "BioLaySumm/"
    "BioLaySumm2025-"
    "LaymanRRG-opensource-track"
)

print(dataset)

for split in dataset.keys():
    print(
        split,
        len(dataset[split]),
        dataset[split].column_names
    )

DatasetDict({
    train: Dataset({
        features: ['source', 'images_path', 'radiology_report', 'layman_report'],
        num_rows: 150454
    })
    validation: Dataset({
        features: ['source', 'images_path', 'radiology_report', 'layman_report'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['source', 'images_path', 'radiology_report', 'layman_report'],
        num_rows: 10537
    })
})
train 150454 ['source', 'images_path', 'radiology_report', 'layman_report']
validation 10000 ['source', 'images_path', 'radiology_report', 'layman_report']
test 10537 ['source', 'images_path', 'radiology_report', 'layman_report']


In [ ]:
if "validation" in dataset:
    EVALUATION_SPLIT = "validation"
elif "test" in dataset:
    EVALUATION_SPLIT = "test"
else:
    raise ValueError(
        "No validation or test split found."
    )

evaluation_dataset = dataset[
    EVALUATION_SPLIT
]

print(
    "Evaluation split:",
    EVALUATION_SPLIT
)

print(
    "Number of samples:",
    len(evaluation_dataset)
)

Evaluation split: validation
Number of samples: 10000


In [ ]:
import random

NUMBER_OF_TEST_SAMPLES = 10
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

sample_size = min(
    NUMBER_OF_TEST_SAMPLES,
    len(evaluation_dataset)
)

selected_indices = random.sample(
    range(len(evaluation_dataset)),
    sample_size
)

print("Selected indices:")
print(selected_indices)

Selected indices:
[1824, 409, 4506, 4012, 3657, 2286, 1679, 8935, 1424, 9674]


In [ ]:
test_results = []

for number, index in enumerate(
    selected_indices,
    start=1
):
    example = evaluation_dataset[index]

    report = example[
        "radiology_report"
    ]

    reference = example.get(
        "layman_report",
        ""
    )

    print("\n" + "=" * 70)
    print(
        f"SAMPLE {number} "
        f"| DATASET INDEX {index}"
    )
    print("=" * 70)

    print("\nRADIOLOGY REPORT")
    print(report)

    print("\nREFERENCE SUMMARY")
    print(reference)

    try:
        result = generate_rag_summary(
            report,
            show_retrieval=True
        )

        generated_summary = result[
            "summary"
        ]

        retrieved_terms = " | ".join(
            result["retrieved_terms"]
        )

        retrieval_scores = " | ".join(
            f"{score:.4f}"
            for score in
            result["retrieval_scores"]
        )

        match_types = " | ".join(
            result["match_types"]
        )

        error_message = ""

    except Exception as error:

        generated_summary = ""
        retrieved_terms = ""
        retrieval_scores = ""
        match_types = ""
        error_message = str(error)

        print("\nERROR")
        print(error_message)

    test_results.append({
        "sample_number": number,
        "dataset_index": index,
        "source": example.get(
            "source",
            ""
        ),
        "radiology_report": report,
        "reference_layman_summary":
            reference,
        "retrieved_terms":
            retrieved_terms,
        "retrieval_scores":
            retrieval_scores,
        "match_types":
            match_types,
        "rag_summary":
            generated_summary,
        "error":
            error_message
    })



SAMPLE 1 | DATASET INDEX 1824

RADIOLOGY REPORT
No clear rib fracture line is observed.

REFERENCE SUMMARY
There is no visible line indicating a broken rib.
RETRIEVED TERMS
- rib fracture (exact): 1.0000

RAG SUMMARY
No clear rib fracture line is visible on the X-ray.

SAMPLE 2 | DATASET INDEX 409

RADIOLOGY REPORT
Posteroanterior chest radiograph shows a high-density projection at the posterior third right costal arch, visualized in a previous study dated 17 07 2009, likely related to a bone island or calcified lymph node. No other relevant findings.

REFERENCE SUMMARY
A chest x-ray shows a high-density area at the back of the right ribcage, which was also seen in a previous study from July 17, 2009. This is likely due to a bone island or a calcified lymph node. There are no other significant findings.
RETRIEVED TERMS
- calcified lymph node (exact): 1.0000

RAG SUMMARY
The patient has a high-density projection at the posterior third of the right costal arch on a chest X-ray, which wa

In [ ]:
import pandas as pd

review_df = pd.DataFrame(
    test_results
)

review_df[
    "hallucination_present"
] = ""

review_df[
    "negation_preserved"
] = ""

review_df[
    "all_findings_covered"
] = ""

review_df[
    "laterality_preserved"
] = ""

review_df[
    "manual_comments"
] = ""

display(
    review_df[
        [
            "sample_number",
            "radiology_report",
            "reference_layman_summary",
            "retrieved_terms",
            "rag_summary"
        ]
    ]
)

,sample_number,radiology_report,reference_layman_summary,retrieved_terms,rag_summary
0,1,No clear rib fracture line is observed.,There is no visible line indicating a broken rib.,rib fracture,No clear rib fracture line is visible on the X...
1,2,Posteroanterior chest radiograph shows a high-...,A chest x-ray shows a high-density area at the...,calcified lymph node,The patient has a high-density projection at t...
2,3,No images of consolidation or significant pulm...,No signs of lung consolidation or significant ...,consolidation | infiltrate | cardiomediastinal...,No images of consolidation or significant area...
3,4,Thoracic aortic elongation. Signs of bibasal h...,The main blood vessel in the chest is stretche...,aortic elongation | bibasal hypoventilation,The aorta appears longer or more curved than u...
4,5,Radiological improvement with reduction of bil...,The x-ray shows an improvement in the conditio...,consolidation,The patient's condition has shown improvement ...
5,6,Chronic pulmonary parenchymal changes with sig...,The lungs show long-term changes with signs of...,air trapping | chronic | costophrenic angle | ...,Chronic pulmonary parenchymal changes with sig...
6,7,No evidence of metastatic pulmonary disease is...,The current study does not show any signs of l...,hila,No evidence of metastatic pulmonary disease is...
7,8,Comparison with the previous study from [date]...,Comparing this to the previous exam from [date...,,The patient has persistent bilateral periphera...
8,9,Compared to the previous control dated 21 10 2...,"Since the last check on October 21, 2012, the ...",atelectasis | hemithorax,The patient's lung condition has shown improve...
9,10,Elevation of the right hemidiaphragm appears t...,"The right side of the diaphragm, which is the ...",,The elevation of the right hemidiaphragm appea...


In [ ]:
OUTPUT_FILE = (
    "rag_manual_review_10_samples.csv"
)

review_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    "Saved:",
    OUTPUT_FILE
)

Saved: rag_manual_review_10_samples.csv


In [ ]:
from google.colab import files

files.download(
    OUTPUT_FILE
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
additional_kb_entries = [
    {
        "id": "rrg_extra_001",
        "term": "bone island",
        "meaning": (
            "A small, localized area of dense bone "
            "seen on an image."
        )
    },
    {
        "id": "rrg_extra_002",
        "term": "costal arch",
        "meaning": (
            "The curved lower border of the rib cage."
        )
    },
    {
        "id": "rrg_extra_003",
        "term": "pectus excavatum",
        "meaning": (
            "A chest-wall shape in which the "
            "breastbone is sunken inward."
        )
    },
    {
        "id": "rrg_extra_004",
        "term": "metastatic pulmonary disease",
        "meaning": (
            "Cancer that has spread to the lungs "
            "from another part of the body."
        )
    },
    {
        "id": "rrg_extra_005",
        "term": "prominent pulmonary hila",
        "meaning": (
            "The central regions of the lungs appear "
            "larger or more noticeable than expected."
        )
    },
    {
        "id": "rrg_extra_006",
        "term": "vascular etiology",
        "meaning": (
            "The finding is considered likely to be "
            "related to blood vessels."
        )
    },
    {
        "id": "rrg_extra_007",
        "term": "bilateral peripheral opacities",
        "meaning": (
            "Areas of increased density are present "
            "along the outer parts of both lungs."
        )
    },
    {
        "id": "rrg_extra_008",
        "term": "alveolar involvement",
        "meaning": (
            "An abnormality affects the small air "
            "spaces within the lungs."
        )
    },
    {
        "id": "rrg_extra_009",
        "term": "retrocardiac atelectasis",
        "meaning": (
            "An area of lung behind the heart is "
            "not fully expanded."
        )
    },
    {
        "id": "rrg_extra_010",
        "term": "granulomatous nodule",
        "meaning": (
            "A small rounded spot formed by a "
            "localized collection of inflammatory "
            "tissue."
        )
    },
    {
        "id": "rrg_extra_011",
        "term": "elevated hemidiaphragm",
        "meaning": (
            "One half of the breathing muscle below "
            "the lungs is positioned higher than "
            "expected."
        )
    },
    {
        "id": "rrg_extra_012",
        "term": "lobulation",
        "meaning": (
            "A structure has a rounded or lobed "
            "contour on the image."
        )
    }
]

In [ ]:
existing_data = (
    vector_store
    ._collection
    .get(
        include=["metadatas"]
    )
)

existing_terms = {
    normalize_text(
        metadata.get("term", "")
    )
    for metadata
    in existing_data["metadatas"]
}

entries_to_add = [
    entry
    for entry in additional_kb_entries
    if normalize_text(
        entry["term"]
    ) not in existing_terms
]

if entries_to_add:

    new_documents = [
        (
            f"Term: {entry['term']}. "
            f"Meaning: {entry['meaning']}"
        )
        for entry in entries_to_add
    ]

    new_metadata = [
        {
            "term": entry["term"],
            "origin":
                "manual_error_analysis",
            "expert_verified": "NO"
        }
        for entry in entries_to_add
    ]

    new_ids = [
        entry["id"]
        for entry in entries_to_add
    ]

    vector_store.add_texts(
        texts=new_documents,
        metadatas=new_metadata,
        ids=new_ids
    )

    print(
        "New entries added:",
        len(entries_to_add)
    )

else:
    print(
        "The additional terms "
        "are already present."
    )

print(
    "Current collection size:",
    vector_store._collection.count()
)

New entries added: 11
Current collection size: 221


In [ ]:
kb_data = (
    vector_store
    ._collection
    .get(
        include=[
            "documents",
            "metadatas"
        ]
    )
)

KB_ENTRIES = []

for document, metadata in zip(
    kb_data["documents"],
    kb_data["metadatas"]
):
    KB_ENTRIES.append({
        "term": metadata.get(
            "term",
            ""
        ),
        "definition": document,
        "metadata": metadata
    })

print(
    "Reloaded KB entries:",
    len(KB_ENTRIES)
)

Reloaded KB entries: 221


In [ ]:
retest_results = []

for number, index in enumerate(
    selected_indices,
    start=1
):
    example = evaluation_dataset[index]

    report = example[
        "radiology_report"
    ]

    reference = example.get(
        "layman_report",
        ""
    )

    print("\n" + "=" * 70)
    print(
        f"RETEST SAMPLE {number} "
        f"| INDEX {index}"
    )
    print("=" * 70)

    print("\nREPORT")
    print(report)

    print("\nREFERENCE")
    print(reference)

    try:
        result = generate_rag_summary(
            report,
            show_retrieval=True
        )

        retest_results.append({
            "sample_number": number,
            "dataset_index": index,
            "radiology_report": report,
            "reference_layman_summary":
                reference,
            "retrieved_terms":
                " | ".join(
                    result[
                        "retrieved_terms"
                    ]
                ),
            "rag_summary":
                result["summary"],
            "error": ""
        })

    except Exception as error:

        retest_results.append({
            "sample_number": number,
            "dataset_index": index,
            "radiology_report": report,
            "reference_layman_summary":
                reference,
            "retrieved_terms": "",
            "rag_summary": "",
            "error": str(error)
        })


RETEST SAMPLE 1 | INDEX 1824

REPORT
No clear rib fracture line is observed.

REFERENCE
There is no visible line indicating a broken rib.
RETRIEVED TERMS
- rib fracture (exact): 1.0000

RAG SUMMARY
Here's the rewritten sentence: No visible break or crack in the ribs is seen. Explanation: - Rib fracture: A break or crack in a rib. - No clear rib fracture line: This means there is no visible mark or line on the rib where the break occurred.

RETEST SAMPLE 2 | INDEX 409

REPORT
Posteroanterior chest radiograph shows a high-density projection at the posterior third right costal arch, visualized in a previous study dated 17 07 2009, likely related to a bone island or calcified lymph node. No other relevant findings.

REFERENCE
A chest x-ray shows a high-density area at the back of the right ribcage, which was also seen in a previous study from July 17, 2009. This is likely due to a bone island or a calcified lymph node. There are no other significant findings.
RETRIEVED TERMS
- calcified l

In [ ]:
retest_df = pd.DataFrame(
    retest_results
)

retest_df.to_csv(
    "rag_retest_10_samples_v2.csv",
    index=False
)

display(
    retest_df[
        [
            "sample_number",
            "radiology_report",
            "retrieved_terms",
            "rag_summary"
        ]
    ]
)

,sample_number,radiology_report,retrieved_terms,rag_summary
0,1,No clear rib fracture line is observed.,rib fracture,Here's the rewritten sentence: No visible brea...
1,2,Posteroanterior chest radiograph shows a high-...,calcified lymph node | bone island | costal arch,Here's the rewritten sentence: The X-ray of th...
2,3,No images of consolidation or significant pulm...,consolidation | infiltrate | cardiomediastinal...,The patient has no visible signs of lung disea...
3,4,Thoracic aortic elongation. Signs of bibasal h...,aortic elongation | bibasal hypoventilation,Here's the rewritten sentence: The aorta appea...
4,5,Radiological improvement with reduction of bil...,consolidation,Here's the rewritten sentence: The patient sho...
5,6,Chronic pulmonary parenchymal changes with sig...,air trapping | chronic | costophrenic angle | ...,Here's the rewritten sentence: The patient has...
6,7,No evidence of metastatic pulmonary disease is...,metastatic pulmonary disease | vascular etiolo...,Here's the rewritten sentence: There is no sig...
7,8,Comparison with the previous study from [date]...,bilateral peripheral opacities,The patient has been showing persistent areas ...
8,9,Compared to the previous control dated 21 10 2...,alveolar involvement | retrocardiac atelectasi...,Here's the rewritten sentence: There has been ...
9,10,Elevation of the right hemidiaphragm appears t...,lobulation,The right diaphragm is slightly higher than us...


In [ ]:
files.download(
    "rag_retest_10_samples_v2.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
version3_results = []

for number, index in enumerate(
    selected_indices,
    start=1
):
    example = evaluation_dataset[index]

    report = example[
        "radiology_report"
    ]

    reference = example.get(
        "layman_report",
        ""
    )

    print("\n" + "=" * 70)
    print(
        f"VERSION 3 — SAMPLE {number} "
        f"| INDEX {index}"
    )
    print("=" * 70)

    print("\nREPORT")
    print(report)

    print("\nREFERENCE")
    print(reference)

    try:
        result = generate_rag_summary(
            report,
            show_retrieval=True
        )

        version3_results.append({
            "sample_number": number,
            "dataset_index": index,
            "source":
                example.get("source", ""),
            "radiology_report": report,
            "reference_layman_summary":
                reference,
            "retrieved_terms":
                " | ".join(
                    result[
                        "retrieved_terms"
                    ]
                ),
            "retrieval_scores":
                " | ".join(
                    f"{score:.4f}"
                    for score in
                    result[
                        "retrieval_scores"
                    ]
                ),
            "match_types":
                " | ".join(
                    result[
                        "match_types"
                    ]
                ),
            "rag_summary":
                result["summary"],
            "error": ""
        })

    except Exception as error:

        version3_results.append({
            "sample_number": number,
            "dataset_index": index,
            "source":
                example.get("source", ""),
            "radiology_report": report,
            "reference_layman_summary":
                reference,
            "retrieved_terms": "",
            "retrieval_scores": "",
            "match_types": "",
            "rag_summary": "",
            "error": str(error)
        })


VERSION 3 — SAMPLE 1 | INDEX 1824

REPORT
No clear rib fracture line is observed.

REFERENCE
There is no visible line indicating a broken rib.
RETRIEVED TERMS
- rib fracture (exact): 1.0000

RAG SUMMARY
A break or crack in a rib.

VERSION 3 — SAMPLE 2 | INDEX 409

REPORT
Posteroanterior chest radiograph shows a high-density projection at the posterior third right costal arch, visualized in a previous study dated 17 07 2009, likely related to a bone island or calcified lymph node. No other relevant findings.

REFERENCE
A chest x-ray shows a high-density area at the back of the right ribcage, which was also seen in a previous study from July 17, 2009. This is likely due to a bone island or a calcified lymph node. There are no other significant findings.
RETRIEVED TERMS
- calcified lymph node (exact): 1.0000
- bone island (exact): 1.0000
- costal arch (exact): 1.0000

RAG SUMMARY
A dense area is visible in the back of the right breastbone (costal arch) on a previous X-ray that was taken 

In [ ]:
version3_df = pd.DataFrame(
    version3_results
)

VERSION3_FILE = (
    "rag_retest_10_samples_v3.csv"
)

version3_df.to_csv(
    VERSION3_FILE,
    index=False
)

display(
    version3_df[
        [
            "sample_number",
            "radiology_report",
            "retrieved_terms",
            "rag_summary"
        ]
    ]
)

,sample_number,radiology_report,retrieved_terms,rag_summary
0,1,No clear rib fracture line is observed.,rib fracture,A break or crack in a rib.
1,2,Posteroanterior chest radiograph shows a high-...,calcified lymph node | bone island | costal arch,A dense area is visible in the back of the rig...
2,3,No images of consolidation or significant pulm...,consolidation | infiltrate | cardiomediastinal...,No fluid or pus is visible in the lungs. The h...
3,4,Thoracic aortic elongation. Signs of bibasal h...,aortic elongation | bibasal hypoventilation,The aorta appears longer or more curved than u...
4,5,Radiological improvement with reduction of bil...,consolidation,"The patient's lungs have improved, and there a..."
5,6,Chronic pulmonary parenchymal changes with sig...,air trapping | chronic | costophrenic angle | ...,Patient has chronic lung damage that makes it ...
6,7,No evidence of metastatic pulmonary disease is...,metastatic pulmonary disease | vascular etiolo...,There is no cancer in the lungs. The lungs hav...
7,8,Comparison with the previous study from [date]...,bilateral peripheral opacities,Bilateral peripheral opacities persisting on c...
8,9,Compared to the previous control dated 21 10 2...,alveolar involvement | retrocardiac atelectasi...,There is improvement in the lungs compared to ...
9,10,Elevation of the right hemidiaphragm appears t...,lobulation,The right hemidiaphragm is elevated and has a ...


In [ ]:
files.download(
    VERSION3_FILE
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>